# MTHexapod Faults per strut

As preparation for the shutdown that will happen on Sep 2025,  
we want to find out what it the strut that is responsible for the major number of faults.  

Relevant documents:
  - [SITCOM-2192 Hexapod Faults per Strut](https://ls.st/SITCOM-2192)
  - [Copley Drives Manual](https://copleycontrols.com/wp-content/uploads/2018/02/All-CANopen_Programmers_Manual-Manual.pdf)
  - [MTHexapod - mthexapod.electrical](https://ts-xml.lsst.io/sal_interfaces/MTHexapod.html#electrical)

The `sal_index` can be either 1 (Camera Hexapod) or 2 (M2 Hexapod).  
The `min_log_level` correspondts to the minimum logging level required to print the messages.  

In [ ]:
start_day_obs = 20250415  # First light
end_day_obs = 20250922  # Start of maintenance shutdown
sal_index = 2
min_log_level = 40

In [ ]:
import enum
import logging
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from datetime import timedelta
from IPython.display import display, HTML
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from pathlib import Path

from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsStartTime,
    makeEfdClient,
)
from lsst.ts.xml.enums.MTHexapod import ApplicationStatus, EnabledSubstate
from lsst_efd_client import EfdClient
from astropy.time import Time
from io import BytesIO
import base64
import requests

In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()

# Global variables
HEXAPOD_AXES = ["X", "Y", "Z", "U", "V", "W"]
N_AXES = len(HEXAPOD_AXES)
N_STRUTS = 6


# Enumerations for convenience
class CopleyStatusWord(enum.IntFlag):
    """
    Status word used in `mthexapod.electrical` as the `copleyStatusWordDrive` column. 
    See page 60 in the Copley Driver's Manual in the link below:

    - https://copleycontrols.com/wp-content/uploads/2018/02/All-CANopen_Programmers_Manual-Manual.pdf
    
    The numbers in the page above are the bit positions, and the values
    are the corresponding powers of 2. For example, the READY_TO_SWITCH_ON
    bit is at position 0, which corresponds to the value 2^0 = 1.
    The SWITCHED_ON bit is at position 1, which corresponds to the value
    2^1 = 2, and so on. 
    
    The values are powers of 2, so they can be combined
    using bitwise OR operations to represent multiple states at once. 
    These values are represented here as hexadecimal values for convenience,
    but they can also be used as decimal values.
    
    The values are also used in the `copleyStatusWordDrive` column in
    `mthexapod.electrical` to indicate the current status of the drive
    in the Copley hexapod system.
    """

    READY_TO_SWITCH_ON = 0x1
    SWITCHED_ON = 0x2
    OPERATION_ENABLED = 0x4
    FAULT = 0x8
    VOLTAGE_ENABLED = 0x10
    QUICK_STOP = 0x20
    SWITCH_ON_DISABLED = 0x40
    WARNING = 0x80
    LAST_TRAJECTORY_ABORTED = 0x100
    REMOTE_ON = 0x200
    TARGET_REACHED = 0x400
    INTERNAL_LIMIT_ACTIVE = 0x800
    SET_POINT_ACK = 0x1000
    FOLLOWING_ERROR = 0x2000
    MOVING = 0x4000
    CAPTURED_HOME_POSITION = 0x8000


class CopleyLatchingFaultStatus(enum.IntFlag):
    """
    Status word used in `mthexapod.electrical` as the `copleyLatchingFaultStatus` column. 
    See page 70 in the Copley Driver's Manual in the link below:

    - https://copleycontrols.com/wp-content/uploads/2018/02/All-CANopen_Programmers_Manual-Manual.pdf
    
    The numbers in the page above are the bit positions, and the values
    are the corresponding powers of 2. For example, the READY_TO_SWITCH_ON
    bit is at position 0, which corresponds to the value 2^0 = 1.
    The SWITCHED_ON bit is at position 1, which corresponds to the value
    2^1 = 2, and so on. 
    
    The values are powers of 2, so they can be combined
    using bitwise OR operations to represent multiple states at once. 
    These values are represented here as hexadecimal values for convenience,
    but they can also be used as decimal values.
    """
   
    FATAL_DATA_FLASH = 0x1
    FATAL_AMPLIFIER_INTERNAL_ERROR = 0x2
    SHORT_CIRCUIT = 0x4
    AMPLIFIER_OVER_TEMPERATURE = 0x8
    MOTOR_OVER_TEMPERATURE = 0x10
    OVER_VOLTAGE = 0x20
    UNDER_VOLTAGE = 0x40
    FEEDBACK_FAULT = 0x80
    PHASING_ERROR = 0x100
    TRACKING_ERROR = 0x200
    OVER_CURRENT = 0x400
    FPGA_FAILURE = 0x800
    INPUT_LOST = 0x1000
    FPGA_FAILURE_TWO = 0x2000
    SAFETY_CIRCUIT_FAULT = 0x4000
    UNABLE_TO_CONTROL_CURRENT = 0x8000


class CopleyFaultStatus(enum.IntFlag):
    """
    Status word used in `mthexapod.electrical` as the `copleyFaultStatus` column. 
    See page 62 in the Copley Driver's Manual in the link below:

    - https://copleycontrols.com/wp-content/uploads/2018/02/All-CANopen_Programmers_Manual-Manual.pdf
    
    The numbers in the page above are the bit positions, and the values
    are the corresponding powers of 2. For example, the READY_TO_SWITCH_ON
    bit is at position 0, which corresponds to the value 2^0 = 1.
    The SWITCHED_ON bit is at position 1, which corresponds to the value
    2^1 = 2, and so on. 
    
    The values are powers of 2, so they can be combined
    using bitwise OR operations to represent multiple states at once. 
    These values are represented here as hexadecimal values for convenience,
    but they can also be used as decimal values.
    """

    SHORT_CIRCUIT = 0x1
    AMPLIFIER_OVER_TEMPERATURE = 0x2
    OVER_VOLTAGE = 0x4
    UNDER_VOLTAGE = 0x8
    MOTOR_TEMPERATURE_SENSOR_ACTIVE = 0x10
    FEEDBACK_ERROR = 0x20
    MOTOR_PHASING_ERROR = 0x40
    CURRENT_OUTPUT_LIMITED = 0x80
    VOLTAGE_OUTPUT_LIMITED = 0x100
    POS_LIMIT_SWITCH_ACTIVE = 0x200
    NEG_LIMIT_SWITCH_ACTIVE = 0x400
    ENABLE_INPUT_NOT_ACTIVE = 0x800
    AMP_DISABLED_BY_SOFTWARE = 0x1000
    TRYING_TO_STOP_MOTOR = 0x2000
    MOTOR_BRAKE_ACTIVE = 0x4000
    PWM_OUTPUT_DISABLED = 0x8000
    POSITIVE_SW_LIMIT_CONDITION = 0x10000
    NEGATIVE_SW_LIMIT_CONDITION = 0x20000
    TRACKING_ERROR = 0x40000
    TRACKING_WARNING = 0x80000
    AMPLIFIER_IN_RESET_CONDITION = 0x100000
    POSITION_WRAPPED = 0x200000  # See the manual for more
    AMPLIFIER_FAULT = 0x400000
    REACHED_VELOCITY_LIMIT = 0x800000
    REACHED_ACCELERATION_LIMIT = 0x1000000
    POSITION_ERROR = 0x2000000
    HOME_SWITCH_IS_ACTIVE = 0x4000000
    IN_MOTION = 0x8000000
    VELOCITY_WINDOW = 0x10000000
    PHASE_NOT_YET_INITIALIZED = 0x20000000
    COMMAND_FAULT = 0x40000000


# Set global font size for labels, titles, and ticks
plt.rcParams.update(
    {
        "axes.grid": True,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "axes.formatter.useoffset": False,
        "axes.formatter.use_mathtext": False,
        "axes.formatter.limits": (-100, 100),
        "figure.figsize": (11, 6),
        "font.size": 12,
        "grid.color": "#b0b0b0",
        "grid.linestyle": ":",
        "grid.linewidth": 0.5,
        "grid.alpha": 0.75,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

level_colors = {
    "D": "#1f77b4",  # blue
    "I": "#2ca02c",  # green
    "W": "#ff7f0e",  # orange
    "E": "#d62728",  # red
    "C": "#9467bd",  # purple
}

## Data Analysis

Let's start defining some helper functions.  
For now, let me focus in querying and understanding the data.  
Remember that we want to know every time that the hexapod faulted and
find out which strut was possibly responsible for this fault.

In [ ]:
async def query_hexapod_controller_state(
    client: EfdClient, start_time: Time, end_time: Time, sal_index: int
) -> pd.DataFrame:
    """
    Query the hexapod controller state between `start_day_obs` and `end_day_obs`
    for a given `sal_index` (1: camera hexapod, 2: m2 hexapod)
    """
    query = f"""
        SELECT time, enabledSubstate, applicationStatus
        FROM "lsst.sal.MTHexapod.logevent_controllerState"
        WHERE time >= '{start_time.isot}Z'
        AND time <= '{end_time.isot}Z'
        AND salIndex = {sal_index}
    """

    _df = await client.influx_client.query(query)
    if not all(_df):
        print(
            "No data found for the specified time range and sal_index. "
            "Returning empty DataFrame."
            )
        return pd.DataFrame()

    _df["applicationStatusName"] = _df["applicationStatus"].apply(
        lambda x: ApplicationStatus(x).name
    )
    _df["enabledSubstateName"] = _df["enabledSubstate"].apply(
        lambda x: EnabledSubstate(x).name
    )

    return _df


async def query_hexapod_electrical(
    client: EfdClient, start_time: Time, end_time: Time, sal_index: int
) -> pd.DataFrame:

    columns = [
        f"copleyStatusWordDrive{i}, copleyLatchingFaultStatus{i}, copleyFaultStatus{i}" 
        for i in range(N_STRUTS)
    ]

    query = f"""
        SELECT {", ".join(columns)}
        FROM "lsst.sal.MTHexapod.electrical"
        WHERE time >= '{start_time.isot}Z'
        AND time <= '{end_time.isot}Z'
        AND salIndex = {sal_index}
        """
        
    _df = await client.influx_client.query(query)
    
    for i in range(N_STRUTS):
        _df[f"copleyStatusWordDrive{i}"] = _df[f"copleyStatusWordDrive{i}"].apply(
            lambda x: CopleyStatusWord(x) if x is not None else None
        )
        _df[f"copleyLatchingFaultStatus{i}"] = _df[f"copleyLatchingFaultStatus{i}"].apply(
            lambda x: CopleyLatchingFaultStatus(x) if x is not None else None
        )
        _df[f"copleyFaultStatus{i}"] = _df[f"copleyFaultStatus{i}"].apply(
            lambda x: CopleyFaultStatus(x) if x is not None else None
        )
    
    return _df


async def query_hexapod_log_messages(
    client: EfdClient, start_day_obs: int, end_day_obs: int, sal_index: int
) -> pd.DataFrame:
    """
    Query error messages from the log messages between `start_day_obs` and
    `end_day_obs` for a given `sal_index` (1: camera hexapod, 2: m2 hexapod)
    """
    start_time = getDayObsStartTime(start_day_obs)
    end_time = getDayObsEndTime(end_day_obs)

    query = f"""
        SELECT functionName, level, lineNumber, message
        FROM "lsst.sal.MTHexapod.logevent_logMessage"
        WHERE time >= '{start_time.isot}Z'
        AND time <= '{end_time.isot}Z'
        AND salIndex = {sal_index}
        AND level >= {min_log_level}
    """

    _df = await client.influx_client.query(query)

    return _df

Now, let me find out what is the size of my dataframe.  
This will affect how I will design the rest of my notebook.

In [ ]:
log_messages_df = await query_hexapod_log_messages(
    efd_client, start_day_obs, end_day_obs, sal_index
)

print(
    "The total number of log messages between {} and {} is {}".format(
        start_day_obs, end_day_obs, log_messages_df.index.size  
    )
)

It is quite a lot.  
I will start with the first error in the data frame and
I will print out a plot with the status for each of the columns in the electrical dataframe. 

In [ ]:
async def get_hexapod_data_around_log(
    log_messages_df : pd.DataFrame, 
    log_index : int, 
    sal_index: int,
    time_window: int=20 
) -> None:
    """
    Given a log_messages_df and a log_index, return controller state and electrical data
    around the log entry within the specified time window.
    """
    time_window = timedelta(seconds=time_window)
    time_reference = Time(log_messages_df.index[log_index])
    time_end = time_reference 
    time_start = time_reference - time_window

    controller_state_df = await query_hexapod_controller_state(
        efd_client, time_start, time_end, sal_index
    )

    electrical_data = await query_hexapod_electrical(
        efd_client, time_start, time_end, sal_index
    )
    
    return controller_state_df, electrical_data

In [ ]:
controller_state_df, electrical_data_df = await get_hexapod_data_around_log(
    log_messages_df, 0, sal_index, time_window=10
)

Let's have a look at our data.  
The events in `controller_state_df` are already decoded.  
This means that you can find out what is going on on each event.  

In [ ]:
controller_state_df.head(10)

The data in the `electrical_data_df` is not translated yet since I want to unpack the enumeration later. 

In [ ]:
electrical_data_df.head(5)